In [ ]:
import numpy as np
import pandas as pd


In [ ]:
ev = pd.read_csv('state_ev_data.csv')
chargers = pd.read_csv('OperationalPC.csv')

In [ ]:
ev = ev.drop(columns=['Charging_Stations'])

In [ ]:
ev.head()

,State,EV_Sales_FY2023_thousands
0,Andaman & Nicobar,0.1
1,Andhra Pradesh,28.7
2,Arunachal Pradesh,0.2
3,Assam,31.8
4,Bihar,45.3


In [ ]:
chargers.head()

,State,No. of Operational PCS
0,Andaman & Nicobar,3
1,Andhra Pradesh,327
2,Arunachal Pradesh,9
3,Assam,86
4,Bihar,124


In [ ]:
ev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 2 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   State                      35 non-null     object 
 1   EV_Sales_FY2023_thousands  35 non-null     float64
dtypes: float64(1), object(1)
memory usage: 692.0+ bytes


In [ ]:
chargers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   State                   34 non-null     object
 1   No. of Operational PCS  34 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 676.0+ bytes


In [ ]:
ev.isnull().sum()

,0
State,0
EV_Sales_FY2023_thousands,0


In [ ]:
chargers.isnull().sum()

,0
State,0
No. of Operational PCS,0


In [ ]:
ev.describe()

,EV_Sales_FY2023_thousands
count,35.000000
mean,31.005714
std,37.005532
min,0.000000
25%,2.850000
50%,16.100000
75%,45.300000
max,158.700000


In [ ]:
chargers.describe()

,No. of Operational PCS
count,34.000000
mean,357.235294
std,617.579099
min,1.000000
25%,18.750000
50%,129.500000
75%,451.250000
max,3079.000000


In [ ]:
chargers.columns = ['State', 'Charging_Stations']

In [ ]:
ev['State'] = ev['State'].replace({
    'Jammu & Kashmir' : 'Jammu and Kashmir',
    'Puducherry'      : 'Pondicherry'})

In [ ]:
india_ev = pd.merge(ev, chargers, on='State', how='left')

In [ ]:
india_ev.head()

,State,EV_Sales_FY2023_thousands,Charging_Stations
0,Andaman & Nicobar,0.1,3.0
1,Andhra Pradesh,28.7,327.0
2,Arunachal Pradesh,0.2,9.0
3,Assam,31.8,86.0
4,Bihar,45.3,124.0


In [ ]:
india_ev['Charging_Stations'] = india_ev['Charging_Stations'].fillna(0)

In [ ]:
india_ev['EV_Sales'] = (india_ev['EV_Sales_FY2023_thousands'] * 1000).astype(int)

In [ ]:
india_ev['EVs_per_Charger'] = (
    india_ev['EV_Sales'] /
    india_ev['Charging_Stations'].replace(0, float('nan'))
).round(1)

In [ ]:
india_ev[['EV_Sales', 'Charging_Stations', 'EVs_per_Charger']].describe()

,EV_Sales,Charging_Stations,EVs_per_Charger
count,35.000000,35.000000,33.000000
mean,31005.685714,347.000000,358.778788
std,37005.530695,611.435004,954.616800
min,0.000000,0.000000,0.000000
25%,2850.000000,17.500000,40.000000
50%,16100.000000,124.000000,73.500000
75%,45300.000000,426.500000,248.100000
max,158700.000000,3079.000000,5216.700000


In [ ]:
# Source: IEA Global EV Outlook 2023 — global average 10 EVs per charger
global_standard = 10
india_avg = round(
    india_ev['EV_Sales'].sum() / india_ev['Charging_Stations'].sum(), 1
)
india_ev['Gap_vs_Global_Standard'] = (
    india_ev['EVs_per_Charger'] - global_standard
).round(1)

india_ev['Gap_Severity'] = pd.cut(
    india_ev['EVs_per_Charger'],
    bins=[0, 10, 50, 200, 99999],
    labels=['At Global Standard', 'Below Standard', 'Critical', 'Extreme Crisis']
)

In [ ]:
print(f"IEA Global Standard:  {global_standard} EVs per charger")
print(f"India Average:        {india_avg} EVs per charger")
print(f"India is {round(india_avg/global_standard, 1)}x worse than global standard")
print(f"\nGap Severity by State:")
print(india_ev['Gap_Severity'].value_counts())
print(f"\nWorst 10 States vs Global Standard:")
print(india_ev[['State', 'EVs_per_Charger', 'Gap_vs_Global_Standard', 'Gap_Severity']]
      .sort_values('Gap_vs_Global_Standard', ascending=False)
      .head(10))


IEA Global Standard:  10 EVs per charger
India Average:        89.4 EVs per charger
India is 8.9x worse than global standard

Gap Severity by State:
Gap_Severity
Critical              13
Below Standard        10
Extreme Crisis         9
At Global Standard     0
Name: count, dtype: int64

Worst 10 States vs Global Standard:
                State  EVs_per_Charger  Gap_vs_Global_Standard    Gap_Severity
5          Chandigarh           5216.7                  5206.7  Extreme Crisis
28             Sikkim           2200.0                  2190.0  Extreme Crisis
25        Pondicherry            695.7                   685.7  Extreme Crisis
3               Assam            369.8                   359.8  Extreme Crisis
11   Himachal Pradesh            365.9                   355.9  Extreme Crisis
4               Bihar            365.3                   355.3  Extreme Crisis
12  Jammu and Kashmir            331.9                   321.9  Extreme Crisis
23           Nagaland            300.0     

In [ ]:
india_ev.to_csv('india_ev.csv', index=False)

from google.colab import files
files.download('india_ev.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>